In [ ]:
# 05 - Section 6 Sparse Gated Calibration Evaluation

This notebook reproduces the sparse gated calibration and evaluation experiment reported in Section 6 of the manuscript.

It uses the Section 5 outputs generated by:

`04_section5_statistical_analysis.ipynb`

Expected inputs:

- `outputs/04_section5_statistical_analysis/tidy_clean.csv`
- `outputs/04_section5_statistical_analysis/delta_temp_R_C40_DR1C_bucket_checked.csv`
- `outputs/04_section5_statistical_analysis/delta_ratehist_T25_R_C40_bucket_checked.csv`

The experiment evaluates four variants:

- Baseline
- Thermal
- RateHist
- Both

The evaluation is a sparse gated correction experiment, not a full profile-level LUT prediction experiment. Corrections are applied only in measured operating contexts and SoC bands where Section 5 detected non-negligible voltage shifts.

In [ ]:
# ============================================================
# Section 6 - Sparse Gated Calibration and Evaluation cell 2
# GitHub/reproducibility-ready single cell
#
# Purpose:
#   Reproduce the sparse gated correction experiment reported in Section 6.
#
# Required upstream notebook:
#   04_section5_statistical_analysis.ipynb
#
# Expected inputs:
#   outputs/04_section5_statistical_analysis/tidy_clean.csv
#   outputs/04_section5_statistical_analysis/delta_temp_R_C40_DR1C_bucket_checked.csv
#   outputs/04_section5_statistical_analysis/delta_ratehist_T25_R_C40_bucket_checked.csv
#
# Terminology:
#   Rate        = charge cut-off current level: C/5, C/40
#   Charge_Rate = discharge-rate/rate-history level: 0.7C, 1C, 2C
#
# Variant labels:
#   Baseline
#   Thermal
#   RateHist
#   Both
#
# Outputs:
#   outputs/05_section6_calibration_evaluation/
# ============================================================

import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    from IPython.display import display
except Exception:
    display = print


# ============================================================
# Paths
# ============================================================

SECTION5_DIR = "outputs/04_section5_statistical_analysis"
OUT_DIR = "outputs/05_section6_calibration_evaluation"
FIG_DIR = os.path.join(OUT_DIR, "revised_figures")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

SEARCH_BASES = [
    SECTION5_DIR,
    ".",
]


def find_file(candidates):
    """
    Locate one required file using repository-relative paths.

    The function first checks each candidate as written, then checks each
    candidate under the configured search bases. This supports both clean
    GitHub execution and notebook reruns from slightly different directories.
    """
    if isinstance(candidates, str):
        candidates = [candidates]

    checked = []

    for candidate in candidates:
        if os.path.exists(candidate):
            return os.path.abspath(candidate)
        checked.append(candidate)

        for base in SEARCH_BASES:
            path = os.path.join(base, candidate)
            checked.append(path)

            if os.path.exists(path):
                return os.path.abspath(path)

    raise FileNotFoundError(
        "Could not locate any required input file. Checked:\n"
        + "\n".join(f" - {p}" for p in checked)
    )


TIDY = find_file([
    "tidy_clean.csv",
    "outputs/04_section5_statistical_analysis/tidy_clean.csv",
])

# Thermal contrast:
#   60C - 25C at charge cut-off current C/40 and discharge-rate/rate-history 1C.
DELTA_TEMP = find_file([
    "delta_temp_R_C40_DR1C_bucket_checked.csv",
    "delta_temp_R_C40_DR1C.csv",
    # Legacy fallbacks from earlier notebook drafts:
    "delta_temp_R_C40_CR1C_bucket_checked.csv",
    "delta_temp_R_C40_CR1C.csv",
    "delta_temp_bucket_checked.csv",
])

# Rate/history contrast:
#   0.7C - 1C at 25C and charge cut-off current C/40.
DELTA_RATEHIST = find_file([
    "delta_ratehist_T25_R_C40_bucket_checked.csv",
    "delta_ratehist_T25_R_C40.csv",
    # Legacy fallbacks from earlier notebook drafts:
    "delta_charge_T25_R_C40.csv",
    "delta_ratehist_bucket_checked.csv",
])

print("Section 6 inputs:")
print(" - tidy_clean     :", TIDY)
print(" - delta_temp     :", DELTA_TEMP)
print(" - delta_ratehist :", DELTA_RATEHIST)
print("Output directory :", os.path.abspath(OUT_DIR))


# ============================================================
# Configuration
# ============================================================

SOC_BINS = [0, 20, 60, 90, 100]
SOC_LABELS = ["0-20", "20-60", "60-90", "90-100"]
USE_BANDS = ["0-20", "20-60", "60-90"]

# Anchor reference:
#   T = 25C
#   Rate = C/40, i.e., charge cut-off current
#   Charge_Rate = 1C, i.e., discharge-rate/rate-history level
ANCHOR = {
    "Temperature": 25,
    "Rate": "C/40",
    "Charge_Rate": "1C",
}

THRESH_SMALL = 0.05
THRESH_MEDIUM = 0.10
THRESH_LARGE = 0.15

VARIANT_ORDER = ["Thermal", "RateHist", "Both"]


# ============================================================
# Helper functions
# ============================================================

def practical_bucket(delta_abs):
    """Assign practical-significance bucket from an absolute voltage shift."""
    if pd.isna(delta_abs):
        return "NA"

    value = float(abs(delta_abs))

    if value >= THRESH_LARGE:
        return "large"
    if value >= THRESH_MEDIUM:
        return "moderate"
    if value >= THRESH_SMALL:
        return "small"

    return "negligible"


def standard_band_value(x):
    """Normalize SoC-band labels to the manuscript convention."""
    s = str(x).strip()

    for ch in ["[", "]", "(", ")"]:
        s = s.replace(ch, "")

    s = s.replace(", ", "-").replace(",", "-")
    s = s.replace("0.0-20.0", "0-20")
    s = s.replace("20.0-60.0", "20-60")
    s = s.replace("60.0-90.0", "60-90")
    s = s.replace("90.0-100.0", "90-100")

    return s


def pick_voltage_col(df):
    """Select the voltage column used for Section 6 evaluation."""
    if "Voltage_clean" in df.columns:
        return "Voltage_clean"

    if "Voltage" in df.columns:
        return "Voltage"

    if "V_median" in df.columns:
        df["Voltage"] = pd.to_numeric(df["V_median"], errors="coerce")
        return "Voltage"

    if "V_q25" in df.columns and "V_q75" in df.columns:
        df["Voltage"] = (
            pd.to_numeric(df["V_q25"], errors="coerce")
            + pd.to_numeric(df["V_q75"], errors="coerce")
        ) / 2.0
        return "Voltage"

    raise KeyError(
        "No voltage column found. Expected one of: "
        "Voltage_clean / Voltage / V_median / V_q25+V_q75."
    )


def load_tidy(path):
    """Load Section 5 tidy_clean.csv and standardize required columns."""
    df = pd.read_csv(path)
    vcol = pick_voltage_col(df)

    required = [vcol, "Temperature", "Rate", "Charge_Rate", "SoC"]
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise KeyError(f"tidy_clean.csv is missing required columns: {missing}")

    df = df.dropna(subset=required).copy()

    df["Temperature"] = pd.to_numeric(df["Temperature"], errors="coerce")
    df["SoC"] = pd.to_numeric(df["SoC"], errors="coerce")
    df[vcol] = pd.to_numeric(df[vcol], errors="coerce")

    df["Rate"] = df["Rate"].astype(str).str.strip()
    df["Charge_Rate"] = df["Charge_Rate"].astype(str).str.strip()

    df = df.dropna(subset=["Temperature", "SoC", vcol]).copy()
    df["Temperature"] = df["Temperature"].astype(int)

    if "SoC_band" in df.columns:
        df["SoC_band"] = df["SoC_band"].apply(standard_band_value)
    elif "soc_band" in df.columns:
        df["SoC_band"] = df["soc_band"].apply(standard_band_value)
    else:
        df["SoC_band"] = pd.cut(
            df["SoC"].clip(0, 100),
            bins=SOC_BINS,
            labels=SOC_LABELS,
            include_lowest=True,
        ).astype(str)

    return df, vcol


def load_delta_table(path):
    """
    Load a Section 5 delta table and recompute the practical bucket from
    mean_diff to avoid stale/manual label inconsistencies.
    """
    df = pd.read_csv(path)

    if "soc_band" in df.columns:
        band_col = "soc_band"
    elif "SoC_band" in df.columns:
        band_col = "SoC_band"
    else:
        raise KeyError(
            f"Missing SoC-band column in {path}: expected soc_band or SoC_band."
        )

    if "mean_diff" not in df.columns:
        raise KeyError(f"Missing mean_diff column in {path}.")

    df["soc_band"] = df[band_col].apply(standard_band_value)
    df = df[df["soc_band"].isin(USE_BANDS)].copy()

    df["mean_diff"] = pd.to_numeric(df["mean_diff"], errors="coerce")
    df["bucket"] = df["mean_diff"].abs().apply(practical_bucket)

    return df


def non_negligible(delta_df):
    """Keep only bands with non-negligible practical voltage shifts."""
    return delta_df[
        ~delta_df["bucket"]
        .astype(str)
        .str.lower()
        .str.contains("negligible", na=False)
    ].copy()


def metrics(values):
    """Return MAE/RMSE/N for a numeric vector."""
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]

    if arr.size == 0:
        return pd.Series({"MAE": np.nan, "RMSE": np.nan, "N": 0})

    return pd.Series({
        "MAE": float(np.mean(np.abs(arr))),
        "RMSE": float(np.sqrt(np.mean(arr ** 2))),
        "N": int(arr.size),
    })


def correction_count(a, b, tol=1e-12):
    """Count evaluation points receiving non-zero correction."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    valid = np.isfinite(a) & np.isfinite(b)

    return int(np.sum(valid & (np.abs(a - b) > tol)))


def band_bias(series_v, df, ref_means):
    """Absolute deviation from anchor band mean."""
    band = df["SoC_band"].astype(str)
    anchor_mu = band.map(ref_means)

    return (series_v - anchor_mu).abs()


def bar_plot_mae(df, title, outfile):
    """Diagnostic MAE bar plot."""
    plt.figure(figsize=(5.2, 3.6))

    x = np.arange(len(df))
    y = df["MAE"].to_numpy(dtype=float)

    plt.bar(x, y)
    plt.xticks(x, df["Variant"].astype(str).tolist(), rotation=0)
    plt.ylabel("MAE (|V - anchor_mean|)")
    plt.title(title, fontsize=11)
    plt.tight_layout()
    plt.savefig(outfile, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    print("Saved plot:", outfile)


# ============================================================
# Load data
# ============================================================

tidy, VCOL = load_tidy(TIDY)

anchor = tidy[
    (tidy["Temperature"] == ANCHOR["Temperature"])
    & (tidy["Rate"] == ANCHOR["Rate"])
    & (tidy["Charge_Rate"] == ANCHOR["Charge_Rate"])
].copy()

if anchor.empty:
    raise RuntimeError(
        "Anchor subset is empty. Expected anchor: "
        "Temperature=25, Rate=C/40, Charge_Rate=1C."
    )

ref_means = anchor.groupby("SoC_band", as_index=True)[VCOL].mean().to_dict()

delta_temp = load_delta_table(DELTA_TEMP)
delta_ratehist = load_delta_table(DELTA_RATEHIST)

corr_temp = (
    non_negligible(delta_temp)[["soc_band", "mean_diff"]]
    .rename(columns={"mean_diff": "dV_temp"})
    .set_index("soc_band")
)

corr_ratehist = (
    non_negligible(delta_ratehist)[["soc_band", "mean_diff"]]
    .rename(columns={"mean_diff": "dV_ratehist"})
    .set_index("soc_band")
)

corr_temp.to_csv(os.path.join(OUT_DIR, "calib_table_thermal.csv"))
corr_ratehist.to_csv(os.path.join(OUT_DIR, "calib_table_ratehist.csv"))

# Legacy-compatible file for older draft references. Do not cite this filename.
corr_ratehist.to_csv(
    os.path.join(OUT_DIR, "calib_table_charge_legacy_do_not_use.csv")
)


# ============================================================
# Apply sparse gated corrections
# ============================================================
# Thermal:
#   delta_temp = V(60C) - V(25C)
#   Map 60C -> 25C anchor by V_cal = V - delta_temp.
#
# RateHist:
#   delta_ratehist = V(0.7C) - V(1C)
#   Map 0.7C -> 1C anchor by V_cal = V - delta_ratehist.
#
# Conservative policy:
#   Corrections are applied only in exact measured contexts and bands from
#   which the corresponding Section 5 contrasts were estimated.

def apply_thermal(df):
    out = df[VCOL].astype(float).copy()

    mask = (
        (df["Temperature"] == 60)
        & (df["Rate"] == ANCHOR["Rate"])
        & (df["Charge_Rate"] == ANCHOR["Charge_Rate"])
        & (df["SoC_band"].isin(corr_temp.index))
    )

    if mask.any():
        adj = df.loc[mask, "SoC_band"].map(corr_temp["dV_temp"])
        out.loc[mask] = out.loc[mask] - adj.to_numpy(dtype=float)

    return out


def apply_ratehist(df, vin=None):
    vin = df[VCOL] if vin is None else vin
    out = vin.astype(float).copy()

    mask = (
        (df["Temperature"] == ANCHOR["Temperature"])
        & (df["Rate"] == ANCHOR["Rate"])
        & (df["Charge_Rate"] == "0.7C")
        & (df["SoC_band"].isin(corr_ratehist.index))
    )

    if mask.any():
        adj = df.loc[mask, "SoC_band"].map(corr_ratehist["dV_ratehist"])
        out.loc[mask] = out.loc[mask] - adj.to_numpy(dtype=float)

    return out


tidy["V_base"] = tidy[VCOL].astype(float)
tidy["V_th"] = apply_thermal(tidy)
tidy["V_ratehist"] = apply_ratehist(tidy)

v_after_th = apply_thermal(tidy)
tidy["V_both"] = apply_ratehist(tidy, vin=v_after_th)


# ============================================================
# Evaluate against anchor band means
# ============================================================

variant_cols = {
    "Baseline": "V_base",
    "Thermal": "V_th",
    "RateHist": "V_ratehist",
    "Both": "V_both",
}

for variant, col in variant_cols.items():
    tidy[f"abs_bias_{col}"] = band_bias(tidy[col], tidy, ref_means)


overall_rows = []

for variant, col in variant_cols.items():
    m = metrics(tidy[f"abs_bias_{col}"])

    overall_rows.append({
        "Variant": variant,
        "MAE": m["MAE"],
        "RMSE": m["RMSE"],
        "N": int(m["N"]),
    })

overall = pd.DataFrame(overall_rows)


bandwise_rows = []

for band in SOC_LABELS:
    group = tidy[tidy["SoC_band"] == band]

    if group.empty:
        continue

    for variant, col in variant_cols.items():
        m = metrics(group[f"abs_bias_{col}"])

        bandwise_rows.append({
            "SoC_band": band,
            "Variant": variant,
            "MAE": m["MAE"],
            "RMSE": m["RMSE"],
            "N": int(m["N"]),
        })

bandwise = pd.DataFrame(bandwise_rows)


# ============================================================
# Ablation and sparse-coverage summary
# ============================================================

base_mae = float(overall.loc[overall["Variant"] == "Baseline", "MAE"].iloc[0])
base_rmse = float(overall.loc[overall["Variant"] == "Baseline", "RMSE"].iloc[0])

coverage = {
    "thermal_applied_N": correction_count(tidy["V_th"], tidy["V_base"]),
    "ratehist_applied_N": correction_count(tidy["V_ratehist"], tidy["V_base"]),
    "both_applied_N": correction_count(tidy["V_both"], tidy["V_base"]),
    "total_rows": int(len(tidy)),
    "note": "Counts are evaluation points receiving non-zero corrections, not battery samples.",
}

ablation_rows = []

for variant, col in variant_cols.items():
    mae = float(overall.loc[overall["Variant"] == variant, "MAE"].iloc[0])
    rmse = float(overall.loc[overall["Variant"] == variant, "RMSE"].iloc[0])

    if variant == "Baseline":
        n_applied = 0
    elif variant == "Thermal":
        n_applied = coverage["thermal_applied_N"]
    elif variant == "RateHist":
        n_applied = coverage["ratehist_applied_N"]
    else:
        n_applied = coverage["both_applied_N"]

    delta_mae_v = base_mae - mae
    delta_rmse_v = base_rmse - rmse

    payout_mV_per_point = np.nan
    payout_uV_per_point = np.nan

    if n_applied > 0:
        payout_mV_per_point = 1000.0 * delta_mae_v / n_applied
        payout_uV_per_point = 1_000_000.0 * delta_mae_v / n_applied

    ablation_rows.append({
        "Variant": variant,
        "Delta_MAE_V": delta_mae_v,
        "Delta_RMSE_V": delta_rmse_v,
        "Delta_MAE_mV": 1000.0 * delta_mae_v,
        "Delta_RMSE_mV": 1000.0 * delta_rmse_v,
        "N_applied": int(n_applied),
        "Payout_per_point_mV_per_point": payout_mV_per_point,
        "Payout_per_point_uV_per_point": payout_uV_per_point,
    })

ablation = pd.DataFrame(ablation_rows)


band_delta_rows = []

for band in SOC_LABELS:
    sub = bandwise[bandwise["SoC_band"] == band]

    if sub.empty or "Baseline" not in set(sub["Variant"]):
        continue

    base_band_mae = float(
        sub.loc[sub["Variant"] == "Baseline", "MAE"].iloc[0]
    )

    row = {"SoC_band": band}

    for variant in ["Thermal", "RateHist", "Both"]:
        if variant in set(sub["Variant"]):
            mae_v = float(sub.loc[sub["Variant"] == variant, "MAE"].iloc[0])
            row[variant] = 1000.0 * (base_band_mae - mae_v)

    band_delta_rows.append(row)

band_delta = pd.DataFrame(band_delta_rows)


# ============================================================
# Save core outputs
# ============================================================

tidy.to_csv(
    os.path.join(OUT_DIR, "tidy_sec6_with_calibrations.csv"),
    index=False
)

overall.to_csv(
    os.path.join(OUT_DIR, "overall_metrics.csv"),
    index=False
)

bandwise.to_csv(
    os.path.join(OUT_DIR, "bandwise_metrics.csv"),
    index=False
)

ablation.to_csv(
    os.path.join(OUT_DIR, "ablation_vs_baseline.csv"),
    index=False
)

band_delta.to_csv(
    os.path.join(OUT_DIR, "bandwise_delta_mae_vs_baseline_mV.csv"),
    index=False
)

with open(os.path.join(OUT_DIR, "coverage_report.json"), "w", encoding="utf-8") as f:
    json.dump(coverage, f, indent=2, ensure_ascii=False)

run_info = {
    "inputs": {
        "tidy": TIDY,
        "delta_temp": DELTA_TEMP,
        "delta_ratehist": DELTA_RATEHIST,
    },
    "outputs": {
        "tidy_sec6_with_calibrations": os.path.join(OUT_DIR, "tidy_sec6_with_calibrations.csv"),
        "overall_metrics": os.path.join(OUT_DIR, "overall_metrics.csv"),
        "bandwise_metrics": os.path.join(OUT_DIR, "bandwise_metrics.csv"),
        "ablation_vs_baseline": os.path.join(OUT_DIR, "ablation_vs_baseline.csv"),
        "bandwise_delta_mae": os.path.join(OUT_DIR, "bandwise_delta_mae_vs_baseline_mV.csv"),
        "coverage_report": os.path.join(OUT_DIR, "coverage_report.json"),
    },
    "terminology": {
        "Rate": "charge cut-off current",
        "Charge_Rate": "discharge-rate/rate-history level",
        "RateHist_variant": "correction based on the mild 0.7C-1C discharge-rate/rate-history contrast",
        "N_applied": "number of evaluation points receiving non-zero corrections",
    },
}

with open(os.path.join(OUT_DIR, "sec6_run_summary.json"), "w", encoding="utf-8") as f:
    json.dump(run_info, f, indent=2, ensure_ascii=False)


# ============================================================
# Diagnostic MAE plots
# ============================================================

bar_plot_mae(
    overall,
    "Overall absolute bias vs anchor (lower is better)",
    os.path.join(OUT_DIR, "bar_overall_mae.png"),
)

for band in USE_BANDS:
    sub = (
        bandwise[bandwise["SoC_band"] == band][["Variant", "MAE"]]
        .reset_index(drop=True)
    )

    if sub.empty:
        continue

    safe_band = band.replace("%", "").replace("/", "-")

    bar_plot_mae(
        sub,
        f"SoC band {band}: absolute bias vs anchor",
        os.path.join(OUT_DIR, f"bar_mae_band_{safe_band}.png"),
    )


# ============================================================
# Revised manuscript figures
# Figure 4: Overall improvement vs baseline
# Figure 5: Band-wise Delta MAE vs baseline
# ============================================================

ablation_plot = ablation[ablation["Variant"] != "Baseline"].copy()
ablation_plot["Variant"] = pd.Categorical(
    ablation_plot["Variant"],
    categories=VARIANT_ORDER,
    ordered=True
)
ablation_plot = ablation_plot.sort_values("Variant")


# Figure 4
fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.4), constrained_layout=True)

metric_specs = [
    ("Delta_MAE_mV", r"$\Delta$MAE improvement (mV)"),
    ("Delta_RMSE_mV", r"$\Delta$RMSE improvement (mV)"),
]

for ax, (col, ylabel) in zip(axes, metric_specs):
    x = np.arange(len(ablation_plot))
    y = ablation_plot[col].to_numpy(dtype=float)

    bars = ax.bar(x, y)

    ax.axhline(0, linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(ablation_plot["Variant"].astype(str).values)
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel.replace(" improvement", ""))
    ax.grid(axis="y", alpha=0.25)

    ymax = max(y) if len(y) else 0
    ax.set_ylim(0, ymax * 1.25 if ymax > 0 else 1)

    for bar, val in zip(bars, y):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{val:.2f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

fig.suptitle("Overall error reduction relative to the baseline anchor", fontsize=12)

fig4_png = os.path.join(FIG_DIR, "fig_sec6_overall_improvement.png")
fig4_pdf = os.path.join(FIG_DIR, "fig_sec6_overall_improvement.pdf")

fig.savefig(fig4_png, dpi=600, bbox_inches="tight")
fig.savefig(fig4_pdf, bbox_inches="tight")
plt.show()
plt.close()

print("Saved revised Figure 4:")
print(" -", fig4_png)
print(" -", fig4_pdf)


# Figure 5
band_order = ["0-20", "20-60", "60-90"]

band_delta_plot = band_delta[band_delta["SoC_band"].isin(band_order)].copy()
band_delta_plot["SoC_band"] = pd.Categorical(
    band_delta_plot["SoC_band"],
    categories=band_order,
    ordered=True
)
band_delta_plot = band_delta_plot.sort_values("SoC_band")

x = np.arange(len(band_delta_plot))
width = 0.25

fig, ax = plt.subplots(figsize=(8.2, 3.8), constrained_layout=True)

for i, variant in enumerate(VARIANT_ORDER):
    offsets = x + (i - 1) * width
    values = band_delta_plot[variant].to_numpy(dtype=float)

    bars = ax.bar(offsets, values, width, label=variant)

    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{val:.2f}",
            ha="center",
            va="bottom",
            fontsize=8,
        )

ax.axhline(0, linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels([
    str(b).replace("-", "--") + r"\%"
    for b in band_delta_plot["SoC_band"]
])
ax.set_ylabel(r"$\Delta$MAE improvement vs. baseline (mV)")
ax.set_xlabel("SoC band")
ax.set_title("Band-wise reduction in absolute bias relative to the baseline anchor")
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False, ncol=3)

ymax = band_delta_plot[VARIANT_ORDER].to_numpy(dtype=float).max()
ax.set_ylim(0, ymax * 1.25 if ymax > 0 else 1)

fig5_png = os.path.join(FIG_DIR, "fig_sec6_bandwise_delta_mae.png")
fig5_pdf = os.path.join(FIG_DIR, "fig_sec6_bandwise_delta_mae.pdf")

fig.savefig(fig5_png, dpi=600, bbox_inches="tight")
fig.savefig(fig5_pdf, bbox_inches="tight")
plt.show()
plt.close()

print("Saved revised Figure 5:")
print(" -", fig5_png)
print(" -", fig5_pdf)


# ============================================================
# Figure-value summary for manuscript checking
# ============================================================

summary_rows = []

for _, row in ablation_plot.iterrows():
    summary_rows.append({
        "figure": "Figure 4",
        "level": "overall",
        "variant": str(row["Variant"]),
        "Delta_MAE_mV": row["Delta_MAE_mV"],
        "Delta_RMSE_mV": row["Delta_RMSE_mV"],
        "N_applied": row["N_applied"],
    })

for _, row in band_delta_plot.iterrows():
    for variant in VARIANT_ORDER:
        summary_rows.append({
            "figure": "Figure 5",
            "level": str(row["SoC_band"]),
            "variant": variant,
            "Delta_MAE_mV": row[variant],
            "Delta_RMSE_mV": np.nan,
            "N_applied": np.nan,
        })

summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(FIG_DIR, "sec6_revised_figure_values.csv")
summary_df.to_csv(summary_path, index=False)


# ============================================================
# Manuscript-level quick checks
# ============================================================

print("\n=== Section 6 - Calibration & Evaluation Summary ===")
print("Outputs directory:", os.path.abspath(OUT_DIR))

print("\n-- Correction tables --")
print("Thermal correction table:")
print(corr_temp.reset_index().to_string(index=False) if not corr_temp.empty else "EMPTY")

print("\nRateHist correction table:")
print(corr_ratehist.reset_index().to_string(index=False) if not corr_ratehist.empty else "EMPTY")

print("\n-- Overall metrics --")
display(overall)

print("\n-- Ablation vs baseline --")
display(ablation)

print("\n-- Band-wise metrics --")
display(bandwise)

print("\n-- Band-wise Delta MAE vs baseline (mV) --")
display(band_delta)

print("\n-- Coverage report --")
print(json.dumps(coverage, indent=2, ensure_ascii=False))

print("\n-- Revised figure values --")
display(summary_df)

print("\nSaved core files:")
for key, value in run_info["outputs"].items():
    print(f" - {key}: {os.path.abspath(value)}")

print("\nSaved revised figure files:")
print(" -", os.path.abspath(fig4_png))
print(" -", os.path.abspath(fig4_pdf))
print(" -", os.path.abspath(fig5_png))
print(" -", os.path.abspath(fig5_pdf))
print(" -", os.path.abspath(summary_path))

In [ ]:
# ============================================================
# Manuscript-level validation checks for Section 6
# ============================================================

expected_overall = {
    "Baseline": {"MAE": 0.130499, "RMSE": 0.164297, "N": 6399},
    "Thermal":  {"MAE": 0.129149, "RMSE": 0.163376, "N": 6399},
    "RateHist": {"MAE": 0.129797, "RMSE": 0.163788, "N": 6399},
    "Both":     {"MAE": 0.128447, "RMSE": 0.162864, "N": 6399},
}

expected_coverage = {
    "thermal_applied_N": 252,
    "ratehist_applied_N": 258,
    "both_applied_N": 510,
    "total_rows": 6399,
}

print("=== Section 6 manuscript-level checks ===")

for variant, exp in expected_overall.items():
    row = overall.loc[overall["Variant"] == variant].iloc[0]

    print(
        f"{variant:8s} | "
        f"MAE={float(row['MAE']):.6f} "
        f"RMSE={float(row['RMSE']):.6f} "
        f"N={int(row['N'])}"
    )

print("\nCoverage:")
print(json.dumps(coverage, indent=2, ensure_ascii=False))

print("\nExpected coverage:")
print(json.dumps(expected_coverage, indent=2, ensure_ascii=False))

print("\nAblation table:")
display(ablation)

print("\nBand-wise Delta MAE vs baseline:")
display(band_delta)

In [ ]:
# ============================================================
# Section 6 revised figures  Cell 3
# Figure 4: Overall improvement vs baseline
# Figure 5: Band-wise Delta MAE vs baseline
# GitHub/reproducibility-ready single cell
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

OUT_DIR = "outputs/05_section6_calibration_evaluation"
FIG_DIR = os.path.join(OUT_DIR, "revised_figures")
os.makedirs(FIG_DIR, exist_ok=True)

ablation_path = os.path.join(OUT_DIR, "ablation_vs_baseline.csv")
band_delta_path = os.path.join(OUT_DIR, "bandwise_delta_mae_vs_baseline_mV.csv")
bandwise_metrics_path = os.path.join(OUT_DIR, "bandwise_metrics.csv")

required_files = [
    ablation_path,
    band_delta_path,
    bandwise_metrics_path,
]

missing_files = [p for p in required_files if not os.path.exists(p)]

if missing_files:
    raise FileNotFoundError(
        "Required Section 6 output files are missing. "
        "Run Cell 2 first. Missing files:\n"
        + "\n".join(missing_files)
    )

# ------------------------------------------------------------
# Load Section 6 outputs
# ------------------------------------------------------------

ablation = pd.read_csv(ablation_path)
band_delta = pd.read_csv(band_delta_path)
bandwise_metrics = pd.read_csv(bandwise_metrics_path)

required_ablation_cols = {
    "Variant",
    "Delta_MAE_mV",
    "Delta_RMSE_mV",
    "N_applied",
}

missing_ablation_cols = required_ablation_cols - set(ablation.columns)

if missing_ablation_cols:
    raise KeyError(
        f"Missing columns in ablation_vs_baseline.csv: {missing_ablation_cols}"
    )

required_band_delta_cols = {
    "SoC_band",
    "Thermal",
    "RateHist",
    "Both",
}

missing_band_delta_cols = required_band_delta_cols - set(band_delta.columns)

if missing_band_delta_cols:
    raise KeyError(
        "Missing columns in bandwise_delta_mae_vs_baseline_mV.csv: "
        f"{missing_band_delta_cols}"
    )

# ------------------------------------------------------------
# Prepare Figure 4 data
# ------------------------------------------------------------

variant_order = ["Thermal", "RateHist", "Both"]

ablation_plot = ablation[ablation["Variant"] != "Baseline"].copy()

ablation_plot["Variant"] = pd.Categorical(
    ablation_plot["Variant"],
    categories=variant_order,
    ordered=True,
)

ablation_plot = ablation_plot.sort_values("Variant")

# ------------------------------------------------------------
# Figure 4: Overall Delta MAE and Delta RMSE improvement
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.4), constrained_layout=True)

metrics = [
    ("Delta_MAE_mV", r"$\Delta$MAE improvement (mV)"),
    ("Delta_RMSE_mV", r"$\Delta$RMSE improvement (mV)"),
]

for ax, (col, ylabel) in zip(axes, metrics):
    x = np.arange(len(ablation_plot))
    y = ablation_plot[col].to_numpy(dtype=float)

    bars = ax.bar(x, y)

    ax.axhline(0, linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(ablation_plot["Variant"].astype(str).values)
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel.replace(" improvement", ""))
    ax.grid(axis="y", alpha=0.25)

    ymax = max(y) if len(y) else 0
    ax.set_ylim(0, ymax * 1.25 if ymax > 0 else 1)

    for bar, val in zip(bars, y):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{val:.2f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

fig.suptitle(
    "Overall error reduction relative to the baseline anchor",
    fontsize=12,
)

fig4_png = os.path.join(FIG_DIR, "fig_sec6_overall_improvement.png")
fig4_pdf = os.path.join(FIG_DIR, "fig_sec6_overall_improvement.pdf")

fig.savefig(fig4_png, dpi=600, bbox_inches="tight")
fig.savefig(fig4_pdf, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Saved Figure 4:")
print(fig4_png)
print(fig4_pdf)

# ------------------------------------------------------------
# Prepare Figure 5 data
# ------------------------------------------------------------

band_order = ["0-20", "20-60", "60-90"]

band_delta_plot = band_delta[
    band_delta["SoC_band"].isin(band_order)
].copy()

band_delta_plot["SoC_band"] = pd.Categorical(
    band_delta_plot["SoC_band"],
    categories=band_order,
    ordered=True,
)

band_delta_plot = band_delta_plot.sort_values("SoC_band")

# ------------------------------------------------------------
# Figure 5: Band-wise Delta MAE improvement
# ------------------------------------------------------------

x = np.arange(len(band_delta_plot))
width = 0.25

fig, ax = plt.subplots(figsize=(8.2, 3.8), constrained_layout=True)

for i, variant in enumerate(variant_order):
    offsets = x + (i - 1) * width
    values = band_delta_plot[variant].to_numpy(dtype=float)

    bars = ax.bar(offsets, values, width, label=variant)

    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{val:.2f}",
            ha="center",
            va="bottom",
            fontsize=8,
        )

ax.axhline(0, linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels([
    str(b).replace("-", "--") + r"\%"
    for b in band_delta_plot["SoC_band"]
])
ax.set_ylabel(r"$\Delta$MAE improvement vs. baseline (mV)")
ax.set_xlabel("SoC band")
ax.set_title(
    "Band-wise reduction in absolute bias relative to the baseline anchor"
)
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False, ncol=3)

ymax = band_delta_plot[variant_order].to_numpy(dtype=float).max()
ax.set_ylim(0, ymax * 1.25 if ymax > 0 else 1)

fig5_png = os.path.join(FIG_DIR, "fig_sec6_bandwise_delta_mae.png")
fig5_pdf = os.path.join(FIG_DIR, "fig_sec6_bandwise_delta_mae.pdf")

fig.savefig(fig5_png, dpi=600, bbox_inches="tight")
fig.savefig(fig5_pdf, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Saved Figure 5:")
print(fig5_png)
print(fig5_pdf)

# ------------------------------------------------------------
# CSV summary for manuscript checking
# ------------------------------------------------------------

summary_path = os.path.join(FIG_DIR, "sec6_revised_figure_values.csv")

summary_rows = []

for _, row in ablation_plot.iterrows():
    summary_rows.append({
        "figure": "Figure 4",
        "level": "overall",
        "variant": str(row["Variant"]),
        "Delta_MAE_mV": row["Delta_MAE_mV"],
        "Delta_RMSE_mV": row["Delta_RMSE_mV"],
        "N_applied": row["N_applied"],
    })

for _, row in band_delta_plot.iterrows():
    for variant in variant_order:
        summary_rows.append({
            "figure": "Figure 5",
            "level": str(row["SoC_band"]),
            "variant": variant,
            "Delta_MAE_mV": row[variant],
            "Delta_RMSE_mV": np.nan,
            "N_applied": np.nan,
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(summary_path, index=False)

print("Summary saved:")
print(summary_path)

display(summary_df)

In [ ]:
# ============================================================
# Section 6 - Optional ablation payout diagnostic
# GitHub/reproducibility-ready single cell
#
# Reads:
#   outputs/05_section6_calibration_evaluation/overall_metrics.csv
#   outputs/05_section6_calibration_evaluation/coverage_report.json
#
# Outputs:
#   outputs/05_section6_calibration_evaluation/ablation_summary.csv
#   outputs/05_section6_calibration_evaluation/ablation_delta_mae_mV.png
#   outputs/05_section6_calibration_evaluation/ablation_payout_per_point_uV.png
#
# Terminology:
#   N_applied counts corrected evaluation points, not battery samples.
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

OUT_DIR = "outputs/05_section6_calibration_evaluation"
os.makedirs(OUT_DIR, exist_ok=True)

overall_csv = os.path.join(OUT_DIR, "overall_metrics.csv")
coverage_json = os.path.join(OUT_DIR, "coverage_report.json")

required_files = [overall_csv, coverage_json]
missing_files = [p for p in required_files if not os.path.exists(p)]

if missing_files:
    raise FileNotFoundError(
        "Required Section 6 output files are missing. "
        "Run the main Section 6 calibration cell first. Missing files:\n"
        + "\n".join(missing_files)
    )

# ------------------------------------------------------------
# Load inputs
# ------------------------------------------------------------

overall = pd.read_csv(overall_csv)

with open(coverage_json, "r", encoding="utf-8") as f:
    coverage = json.load(f)

required_overall_cols = {"Variant", "MAE", "RMSE", "N"}
missing_cols = required_overall_cols - set(overall.columns)

if missing_cols:
    raise KeyError(
        f"Missing required columns in overall_metrics.csv: {missing_cols}"
    )

overall["Variant"] = overall["Variant"].astype(str).str.strip()

# Defensive compatibility with older draft labels.
overall["Variant"] = overall["Variant"].replace({
    "Charge": "RateHist",
    "Charge-only": "RateHist",
    "RateHist-only": "RateHist",
    "Thermal-only": "Thermal",
})

if "ratehist_applied_N" not in coverage and "charge_applied_N" in coverage:
    coverage["ratehist_applied_N"] = coverage["charge_applied_N"]

# ------------------------------------------------------------
# Baseline reference
# ------------------------------------------------------------

if "Baseline" not in set(overall["Variant"]):
    raise ValueError("Baseline row not found in overall_metrics.csv.")

baseline_row = overall.loc[overall["Variant"] == "Baseline"].iloc[0]

mae_base = float(baseline_row["MAE"])
rmse_base = float(baseline_row["RMSE"])

variant_order = ["Thermal", "RateHist", "Both"]

def applied_n_for_variant(variant):
    if variant == "Thermal":
        return int(coverage.get("thermal_applied_N", 0))
    if variant == "RateHist":
        return int(coverage.get("ratehist_applied_N", 0))
    if variant == "Both":
        return int(coverage.get("both_applied_N", 0))
    return 0

# ------------------------------------------------------------
# Ablation summary
# ------------------------------------------------------------

records = []

for variant in variant_order:
    if variant not in set(overall["Variant"]):
        raise ValueError(f"{variant} row not found in overall_metrics.csv.")

    row = overall.loc[overall["Variant"] == variant].iloc[0]

    delta_mae_v = mae_base - float(row["MAE"])
    delta_rmse_v = rmse_base - float(row["RMSE"])
    n_applied = applied_n_for_variant(variant)

    payout_mV_per_point = np.nan
    payout_uV_per_point = np.nan

    if n_applied > 0:
        payout_mV_per_point = 1000.0 * delta_mae_v / n_applied
        payout_uV_per_point = 1_000_000.0 * delta_mae_v / n_applied

    records.append({
        "Variant": variant,
        "Delta_MAE_V": delta_mae_v,
        "Delta_RMSE_V": delta_rmse_v,
        "Delta_MAE_mV": 1000.0 * delta_mae_v,
        "Delta_RMSE_mV": 1000.0 * delta_rmse_v,
        "N_applied": n_applied,
        "Payout_per_point_mV_per_point": payout_mV_per_point,
        "Payout_per_point_uV_per_point": payout_uV_per_point,
    })

ablation_summary = pd.DataFrame.from_records(records)

summary_csv = os.path.join(OUT_DIR, "ablation_summary.csv")
ablation_summary.to_csv(summary_csv, index=False)

print("=== Ablation Summary vs Baseline ===")
display(ablation_summary)

print("Saved:")
print(summary_csv)

# ------------------------------------------------------------
# Plot helpers
# ------------------------------------------------------------

def bar_plot(xlabels, y, title, ylabel, outfile):
    plt.figure(figsize=(5.2, 3.6))

    x = np.arange(len(xlabels))
    y = np.asarray(y, dtype=float)

    bars = plt.bar(x, y)

    plt.xticks(x, xlabels)
    plt.ylabel(ylabel)
    plt.title(title, fontsize=11)
    plt.grid(axis="y", alpha=0.25)

    ymax = np.nanmax(y) if len(y) else 0
    plt.ylim(0, ymax * 1.25 if ymax > 0 else 1)

    for bar, val in zip(bars, y):
        if np.isfinite(val):
            plt.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(),
                f"{val:.2f}",
                ha="center",
                va="bottom",
                fontsize=9,
            )

    plt.tight_layout()
    plt.savefig(outfile, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    print("Saved plot:", outfile)

# ------------------------------------------------------------
# Diagnostic plots
# ------------------------------------------------------------

delta_mae_png = os.path.join(OUT_DIR, "ablation_delta_mae_mV.png")
payout_uV_png = os.path.join(OUT_DIR, "ablation_payout_per_point_uV.png")

bar_plot(
    ablation_summary["Variant"].tolist(),
    ablation_summary["Delta_MAE_mV"].to_numpy(dtype=float),
    "Ablation: MAE improvement",
    r"$\Delta$MAE improvement vs. Baseline (mV)",
    delta_mae_png,
)

bar_plot(
    ablation_summary["Variant"].tolist(),
    ablation_summary["Payout_per_point_uV_per_point"].to_numpy(dtype=float),
    "Payout per corrected evaluation point",
    r"$\mu$V per corrected evaluation point",
    payout_uV_png,
)

# ------------------------------------------------------------
# Coverage report
# ------------------------------------------------------------

print("\nCoverage used:")
print(json.dumps(coverage, indent=2, ensure_ascii=False))

print("\nInterpretation note:")
print(
    "N_applied counts evaluation points receiving non-zero sparse corrections. "
    "It is not a number of battery samples or independent cells."
)